In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sb 


In [2]:
## Loading the agriculture dataset 
df = pd.read_csv("../data/raw/India Agriculture Crop Production.csv")

In [3]:
df.head()

,State,District,Crop,Year,Season,Area,Area Units,Production,Production Units,Yield
0,Andaman and Nicobar Islands,NICOBARS,Arecanut,2001-02,Kharif,1254.0,Hectare,2061.0,Tonnes,1.643541
1,Andaman and Nicobar Islands,NICOBARS,Arecanut,2002-03,Whole Year,1258.0,Hectare,2083.0,Tonnes,1.655803
2,Andaman and Nicobar Islands,NICOBARS,Arecanut,2003-04,Whole Year,1261.0,Hectare,1525.0,Tonnes,1.209358
3,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,Arecanut,2001-02,Kharif,3100.0,Hectare,5239.0,Tonnes,1.690000
4,Andaman and Nicobar Islands,SOUTH ANDAMANS,Arecanut,2002-03,Whole Year,3105.0,Hectare,5267.0,Tonnes,1.696296


In [4]:
df.tail()

,State,District,Crop,Year,Season,Area,Area Units,Production,Production Units,Yield
345402,Manipur,IMPHAL WEST,NaN,2019-20,Rabi,NaN,Hectare,NaN,Tonnes,NaN
345403,Manipur,SENAPATI,NaN,2019-20,Rabi,NaN,Hectare,NaN,Tonnes,NaN
345404,Manipur,TAMENGLONG,NaN,2019-20,Rabi,NaN,Hectare,NaN,Tonnes,NaN
345405,Manipur,THOUBAL,NaN,2019-20,Rabi,NaN,Hectare,NaN,Tonnes,NaN
345406,Manipur,UKHRUL,NaN,2019-20,Rabi,NaN,Hectare,NaN,Tonnes,NaN


## Basic preprocessing for making it ready for EDA

In [5]:
## Missing values 
df.isnull().sum()

State                  0
District               0
Crop                  32
Year                   0
Season                 1
Area                  33
Area Units             0
Production          4993
Production Units       0
Yield                 33
dtype: int64

In [6]:
## dropping the rows with missing values 
df.dropna(inplace=True)

In [7]:
df.tail()

,State,District,Crop,Year,Season,Area,Area Units,Production,Production Units,Yield
345370,West Bengal,PURBA BARDHAMAN,Wheat,2000-01,Rabi,6310.0,Hectare,15280.0,Tonnes,2.421553
345371,West Bengal,PURULIA,Wheat,1997-98,Rabi,1895.0,Hectare,2760.0,Tonnes,1.456464
345372,West Bengal,PURULIA,Wheat,1998-99,Rabi,3736.0,Hectare,5530.0,Tonnes,1.480193
345373,West Bengal,PURULIA,Wheat,1999-00,Rabi,2752.0,Hectare,6928.0,Tonnes,2.517442
345374,West Bengal,PURULIA,Wheat,2000-01,Rabi,2979.0,Hectare,7430.0,Tonnes,2.494126


In [8]:
df.dtypes

State                   str
District                str
Crop                    str
Year                    str
Season                  str
Area                float64
Area Units              str
Production          float64
Production Units        str
Yield               float64
dtype: object

In [9]:
## Check for the count for duplicate data 
df.duplicated().sum()

## No duplicate row exists in this dataset

np.int64(0)

In [10]:
#Extract the year ending 
df[['start','end']] = df['Year'].str.split('-',expand=True)

df['start'] = df['start'].astype(int)
df['end'] = df['end'].astype(int)

df['Year_end'] = (df['start'] // 100) * 100 +df['end']

In [11]:
df.drop(columns='Year',inplace=True)
df.drop(columns='end',inplace=True)
df.drop(columns='start',inplace=True)

In [12]:
df.head()

,State,District,Crop,Season,Area,Area Units,Production,Production Units,Yield,Year_end
0,Andaman and Nicobar Islands,NICOBARS,Arecanut,Kharif,1254.0,Hectare,2061.0,Tonnes,1.643541,2002
1,Andaman and Nicobar Islands,NICOBARS,Arecanut,Whole Year,1258.0,Hectare,2083.0,Tonnes,1.655803,2003
2,Andaman and Nicobar Islands,NICOBARS,Arecanut,Whole Year,1261.0,Hectare,1525.0,Tonnes,1.209358,2004
3,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,Arecanut,Kharif,3100.0,Hectare,5239.0,Tonnes,1.690000,2002
4,Andaman and Nicobar Islands,SOUTH ANDAMANS,Arecanut,Whole Year,3105.0,Hectare,5267.0,Tonnes,1.696296,2003


In [13]:
df['Area Units'].unique()

<StringArray>
['Hectare']
Length: 1, dtype: str

In [14]:
df.drop(columns='Area Units',inplace=True)

In [15]:
df['Production Units'].unique()

<StringArray>
['Tonnes', 'Nuts', 'Bales']
Length: 3, dtype: str

In [16]:
df.shape

(340414, 9)

In [17]:
## Applied EDA in this Data set 
df.to_csv("../data/preprocessed/cleaned_agridata.csv")

## Further Preprocessing and Modelling based on the EDA results 

## Train test split 

In [18]:
train = df[(df['Year_end']>=1998) & (df['Year_end']<= 2017)]
test = df[(df['Year_end']>=2018) & (df['Year_end']<= 2020)]

X_train = train.drop('Yield', axis=1)
y_train = train['Yield']

X_test = test.drop('Yield', axis=1)
y_test = test['Yield']


### Season Data Cleaning 

In [19]:
df['Season'].unique()

<StringArray>
['Kharif', 'Whole Year', 'Rabi', 'Autumn', 'Summer', 'Winter']
Length: 6, dtype: str

In [20]:
df['Season'].value_counts()


Season
Kharif        136165
Rabi           99805
Whole Year     67265
Summer         21974
Winter          8238
Autumn          6967
Name: count, dtype: int64

In [21]:
## Whole year represents aggregated annual data and 
# may behave differently from seasonal records 

### Log transformation 
  1. Reducing the skewness 
  2. Reducing the impact of Outliers 
  

In [21]:
df['Area'] = np.log1p(df['Area'])
df['Production'] = np.log1p(df['Production'])
df['Yield'] = np.log1p(df['Yield'])


### Handling Production Units 

In [22]:
df = df[df['Production Units']!='Nuts']

In [23]:
## Converting the Unit of Bales of Cotton , Jute and Mesta into Tonnes as per the indian standard Unit

df[df['Production Units'] == 'Bales']['Crop'].unique()

df.loc[(df['Production Units']=='Bales')&(df['Crop']=='Cotton(lint)'),'Production'] *= 0.17
df.loc[(df['Production Units']=='Bales')&(df['Crop'].isin(['Jute','Mesta'])),'Production'] *= 0.18



In [24]:
df['Yield'] = df['Production'] / df['Area']
df = df.drop('Production Units', axis=1)

### Standard Scaling and encoding of the data with pipeline 


In [25]:
from sklearn.preprocessing import StandardScaler 
scaler = StandardScaler()

In [26]:
! pip install category_encoders



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
## Defining columns for 
numeric_cols = ['Area','Production','Year_end']
ohe_cols = ['Crop','Season','State']
target_enc_cols = ['District']

In [28]:
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline 
from category_encoders.target_encoder import TargetEncoder 

In [29]:
## Build preprocessor 
preprocessor = ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),numeric_cols),
        ('ohe',OneHotEncoder(handle_unknown='ignore'),ohe_cols),
        ('target',TargetEncoder(),target_enc_cols)
    ]
)

In [30]:
! pip install xgboost



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

models = {
    "Linear Regression": LinearRegression(),

    "Ridge": Ridge(alpha=1.0),

    "Lasso": Lasso(alpha=0.01, max_iter=5000),

    "Random Forest": RandomForestRegressor(
        n_estimators=50,
        max_depth=15,
        n_jobs=-1,
        verbose=1,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        n_jobs=-1,
        verbosity=1,
        random_state=42
    )
}


## Cross Validation 
 1. Helps to find the best model based on how cv scores for the test sets.
 2. Reduces variance
 3. Give Stability insight
 4. More reliable model comparison 

In [38]:
from sklearn.model_selection import cross_val_score 
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

results = {}

best_model = None;
best_r2 = -1;

for name, model in models.items():
    
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')


    mean_r2 = cv_scores.mean()
    std_r2 = cv_scores.std()
    
    results[name] = {
        "CV Mean R2": mean_r2,
        "CV Std R2": std_r2
    }

    if mean_r2>best_r2:
       best_r2 = mean_r2
       best_model = pipe
    
    
    
results


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:   24.5s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   44.7s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done  50 out of  50 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:   22.5s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   43.5s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done  50 out of  50 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:   26

{'Linear Regression': {'CV Mean R2': np.float64(0.785663564870549),
  'CV Std R2': np.float64(0.038892441315990386)},
 'Ridge': {'CV Mean R2': np.float64(0.7854009847904779),
  'CV Std R2': np.float64(0.038632020725876966)},
 'Lasso': {'CV Mean R2': np.float64(0.7856825666065956),
  'CV Std R2': np.float64(0.03887557782994148)},
 'Random Forest': {'CV Mean R2': np.float64(0.9541394952167019),
  'CV Std R2': np.float64(0.012054104199082448)},
 'XGBoost': {'CV Mean R2': np.float64(0.9239583939188559),
  'CV Std R2': np.float64(0.020801836833953202)}}

In [39]:
best_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [40]:
best_r2

np.float64(0.9541394952167019)

### Hyperparameter tuning with Random SearchCv

In [41]:
param_dist = {
    'regressor__n_estimators': [100, 150, 200, 300],
    'regressor__max_depth': [10, 15, 20, None],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': ['sqrt', 'log2']
}


In [42]:
from sklearn.model_selection import RandomizedSearchCV 

random_search = RandomizedSearchCV(
    estimator = best_model,
    param_distributions = param_dist,
    n_iter = 15,
    cv=3,
    scoring = 'r2',
    n_jobs=-1,
    verbose = 2,
    random_state = 42
)

In [ ]:
random_search.fit(X_train,y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
